In [1]:
#Modules

import h5py
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt

In [ ]:
# assigning filepath for data

filepath_200GeV = '/Users/justin/code/trajectum/cs_analysis/src/200GeV_200_MeVpTThresholds_SMASH 1.h5'


In [ ]:
# this isnt going to be used


# sub-directories for data sets
# the idea is to start with pions and extend the methodology to kaons and protons


# PION
meanptchargedpion_values = '/meanptchargedpion/STARTPC2/centralitybinned/values'
meanptchargedpion_Uerr = '/meanptchargedpion/STARTPC2/centralitybinned/uppererrors'
meanptchargedpion_Lerr = '/meanptchargedpion/STARTPC2/centralitybinned/lowererrors'

multiplicitychargedpion_value = '/multiplicitychargedpion/STARTPC2/centralitybinned/values'
multiplicitychargedpion_Uerr = '/multiplicitychargedpion/STARTPC2/centralitybinned/uppererrors'
multiplicitychargedpion_Lerr = '/multiplicitychargedpion/STARTPC2/centralitybinned/lowererrors'

# KAON next etc

In [ ]:
# here im going to grab thermal photon data

pt_bins_dir = "multiplicitythermalphotonptbinned/STARTPC200MeV/centralitybinned/bin"
pt_dbins_dir = "multiplicitythermalphotonptbinned/STARTPC200MeV/centralitybinned/dbin"
multiplicitythermalphoton_vals_dir = "multiplicitythermalphotonptbinned/STARTPC200MeV/centralitybinned/values"
multiplicitythermalphoton_lerr_dir = "multiplicitythermalphotonptbinned/STARTPC200MeV/centralitybinned/lowererrors"
multiplicitythermalphoton_uerr_dir = "multiplicitythermalphotonptbinned/STARTPC200MeV/centralitybinned/uppererrors"

with h5py.File(filepath_200GeV, "r") as hdf:
    multiplicitythermalphoton_vals_200GeV = hdf[multiplicitythermalphoton_vals_dir][:]
    multiplicitythermalphoton_lerr_200GeV = hdf[multiplicitythermalphoton_lerr_dir][:]
    multiplicitythermalphoton_uerr_200GeV = hdf[multiplicitythermalphoton_uerr_dir][:]
    pt_bins = hdf[pt_bins_dir][:]
    pt_dbins = hdf[pt_dbins_dir][:]
    centrality=hdf["centrality"][:]

In [6]:
# Importing from filepaths
with h5py.File(filepath1, "r") as hdf:
    pion_values = hdf[meanptchargedpion_values][:]

    # symerr
    pion_Uerr = hdf[meanptchargedpion_Uerr][:]
    pion_Lerr = hdf[meanptchargedpion_Lerr][:]
    pion_Symerr = 0.5 * (pion_Uerr + pion_Lerr)

    # pion multiplicity
    multiplicitypion_values = hdf[multiplicitychargedpion_value][:]

    # symerr 
    multiplicitypion_Uerr = hdf[multiplicitychargedpion_Uerr][:]
    multiplicitypion_Lerr = hdf[multiplicitychargedpion_Lerr][:]
    multiplicitypion_Symerr = 0.5 * (multiplicitypion_Uerr + multiplicitypion_Lerr)
    

In [9]:
# normalizing

# normalizing the results for the 5% bin
pion_values_normal = pion_values / pion_values[16]
pion_symerr_normal = pion_values_normal * np.sqrt(
    (pion_Symerr/pion_values)**2 +
    (pion_Symerr[16]/pion_values[16])**2
)

multiplicitypion_val_norm = multiplicitypion_values / multiplicitypion_values[16]
multiplicitypion_Symerr_norm = multiplicitypion_val_norm * np.sqrt(
    (multiplicitypion_Symerr/multiplicitypion_values)**2 +
    (multiplicitypion_Symerr[16]/multiplicitypion_values[16])**2
)

In [ ]:
def compute_chi2_ndf(xdata, ydata, yerr, fit_func, popt):
    model_y = fit_func(xdata, *popt)
    residuals = (ydata - model_y) / yerr
    chi2 = np.sum(residuals**2)
    ndf = len(xdata) - len(popt)
    chi2_ndf = chi2 / ndf
    return chi2, ndf, chi2_ndf

#Non-analytic model.
def power_law_fit(x,a,cs2):
    return a * x ** cs2